# xgov: full functional dump

Loads `xgov-db`, runs the dataflow-aware constant-propagation pass and the dead-constant elimination pass, then prints the resulting functional representation.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# Pin the Linux codeql binary (VS Code on WSL may pick the Windows one).
os.environ.setdefault("CODEQL", "/home/argi/tools/codeql/codeql")

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import teal_ssa

In [2]:
DB = HERE.parent / "test-dbs" / "xgov-db"

p = teal_ssa.SSAProgram(DB)
p.propagate_constants()
p.propagate_scratch_constants()
p.eliminate_dead_constants()

print(f"{len(p)} assignments, {len(p.vars)} SSA vars, {len(p.phis)} phis, {len(p.blocks)} BBs")

797 assignments, 823 SSA vars, 421 phis, 161 BBs


In [3]:
print(p.functional())

L   2: intcblock 0 1 10 3 ()
L   3: bytecblock 0x766f74655f74797065 0x 0x6f75616964 0x766f74655f6964 0x6f7074696f6e5f636f756e7473 0x69735f626f6f747374726170706564 0x766f7465725f636f756e74 0x636c6f73655f74696d65 0x746f74616c5f6f7074696f6e73 0x56 0x736e617073686f745f7075626c69635f6b6579 0x6d657461646174615f697066735f636964 0x73746172745f74696d65 0x656e645f74696d65 0x71756f72756d 0x6e66745f696d6167655f75726c 0x4c6bea72 0x151f7c75 0x6e66745f61737365745f6964 0x068101 0x2c ()
L   4: V#1@L4 = txn NumAppArgs ()
L   6: V#1@L6 = == (0, V#1@L4)
L   7: bnz main_l14 (V#1@L6)
L   8: V#1@L8 = txna ApplicationArgs 0 ()
L  10: V#1@L10 = == (0x101cea00, V#1@L8)
L  11: bnz main_l13 (V#1@L10)
L  12: V#1@L12 = txna ApplicationArgs 0 ()
L  14: V#1@L14 = == (0x5d4cf066, V#1@L12)
L  15: bnz main_l12 (V#1@L14)
L  16: V#1@L16 = txna ApplicationArgs 0 ()
L  18: V#1@L18 = == (0xa4e8d164, V#1@L16)
L  19: bnz main_l11 (V#1@L18)
L  20: V#1@L20 = txna ApplicationArgs 0 ()
L  22: V#1@L22 = == (0x9546e10f, V#1@L20)
L  

## Range analysis

`propagate_ranges()` is an independent pass that tags SSA vars with a static integer range and type. For now it seeds only from boolean-returning ops (`<`, `>`, `==`, `&&`, …) — their single output gets `range = [0..1]`, `type = uint64` — and unions through phis to fixed point. Rendering with `show_ranges=True` attaches a `/*[var<=1]*/` annotation next to each annotated var.


In [4]:
p.propagate_ranges()

n_var = sum(1 for v in p.vars.values() if v.range is not None)
n_phi = sum(1 for ph in p.phis.values() if ph.range is not None)
print(f"{n_var}/{len(p.vars)} SSA vars have a range, {n_phi}/{len(p.phis)} phis have a range")


131/823 SSA vars have a range, 2/421 phis have a range


Functional dump for the dispatcher prologue (lines 1–30) with range annotations turned on. Note the comparison results (`==` outputs) decorated with `/*[V<=1]*/`, and how the same annotation follows them into the `bnz` consumers.


In [ ]:
p.propagate_stack_shuffles()
p.materialize_phis()
print(p.functional(show_ranges=True))
